In [8]:
import pandas as pd
import re

In [10]:
df = pd.read_csv('/content/all_labelled_cleaned.csv')
df2 = pd.read_csv('/content/training_data_binary.csv')

In [20]:
prompt_col = 'prompt'
placeholder_patterns = [
    r'\binsert\b',
    r'\bname\b',
    r'\busername\b',
    r'\bplaceholder\b',
    r'\bsubject\b',
    r'\btext here\b',
    r'\bXXX\b',
]

In [21]:
pattern = re.compile('|'.join(placeholder_patterns), re.IGNORECASE)

In [22]:
def has_placeholder(text):
    if isinstance(text, str):
        return bool(pattern.search(text))
    return False

In [23]:
df_placeholders = df[df[prompt_col].apply(has_placeholder)].copy()
df_clean = df[~df[prompt_col].apply(has_placeholder)].copy()

In [24]:
df_placeholders.to_csv('placeholder_prompts.csv', index=False)
df_clean.to_csv('clean_prompts.csv', index=False)

In [25]:
print(f"Found {len(df_placeholders)} placeholder prompts out of {len(df)} total.")

Found 378 placeholder prompts out of 998 total.


In [ ]:
df_placeholders.head(20)

In [ ]:
df_placeholders['prompt'][1]

In [35]:
import random

default_fills = {
    'INPUT': 'user input text',
    'USER': 'user',
    'USERNAME': 'jacob',
    'PROMPT CREATOR': 'prompt engineer',
    'HEADER': 'System initialization header',
    'PURPOSE': 'general conversation',
    'CHARACTER': 'fictional AI persona',
    'NAME': 'Alex',
    'SUBJECT': 'topic of discussion',
    'CITY': 'Sydney',
}

In [45]:
def fill_prompt(text):
    if not isinstance(text, str):
        return text
    filled = text

    # 1️⃣ Handle bracket-style placeholders [INPUT], {USER}, <PROMPT CREATOR>
    def replace_bracket(m):
        key = m.group(1).strip().upper()
        return default_fills.get(key, f"{key.lower()}_filled")
    filled = re.sub(r"[\[\{<]\s*([^>\]\}]+?)\s*[\]\}>]", replace_bracket, filled, flags=re.IGNORECASE)

    # 2️⃣ Handle "(KEY)=[...]" and even if closing bracket is missing
    def replace_assignment(m):
        key = m.group(1).strip().upper()
        replacement = default_fills.get(key, f"{key.lower()}_filled")
        return f"+:({key})=[{replacement}]"
    filled = re.sub(r"\+:\s*\(?\s*([A-Za-z _]+?)\s*\)?\s*=\s*\[.*?(?=\n|\Z)", replace_assignment, filled, flags=re.IGNORECASE|re.DOTALL)

    # 3️⃣ Handle "Insert ___ here"
    def replace_insert(m):
        key = m.group(1).strip().upper()
        return default_fills.get(key, f"{key.lower()}_filled")
    filled = re.sub(r"insert\s+(.*?)\s+here", replace_insert, filled, flags=re.IGNORECASE)

    return filled

In [51]:
df["filled_prompt"] = df[prompt_col].apply(fill_prompt)
df.to_csv('filled_jailbreak_prompts.csv', index=False)

In [ ]:
print("Filled")
print(df[['prompt', 'filled_prompt']].head(5))

In [53]:
df_filled = pd.read_csv('/content/filled_jailbreak_prompts.csv')

In [ ]:
df_filled['filled_prompt'][1]

In [ ]:
df_placeholders['prompt'][4]

In [ ]:
df_placeholders['prompt']